# 1、 结构化输出概述
## 1.1 什么是结构化输出  要求模型最终返回一个符合预定义结构的数据对象，例如固定字段的 JSON 、 Pydantic 模型、 TypedDict ，而不再是无格式的自然语言文本

In [ ]:

import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)

# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务
model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
      extra_body={"thinking": {"type": "disabled"}}  # 关键：永久关闭思考模式
)
from langchain.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

{
      "title": " 盗梦空间 ",
      "year": 2010,
      "director": " 克里斯托弗 · 诺兰 ",
      "rating": 9.3
}

## 1.2 传统方式 vs 结构化输出

In [ ]:
import json
from dataclasses import dataclass
from langchain_deepseek import ChatDeepSeek  # 如果你的环境不一样，改这里


# --- 补全部分开始 ---
# 1. 自己定义 Person
@dataclass
class Person:
      name: str
      age: int
      occupation: str


# 2. 初始化你的模型
model = ChatDeepSeek(model="deepseek-chat")
# --- 补全部分结束 ---


# ======== 下面是截图里的原版代码 ========
# 1. 提示词要求 JSON
prompt = " 以 JSON 格式返回： {name, age, occupation}"
response = model.invoke(prompt)

# 2. 手动解析
data = json.loads(response.content)

# 3. 手动验证类型
if not isinstance(data['age'], int):
      raise ValueError("age must be int")

# 4. 手动创建对象
person = Person(**data)

print("成功输出：", person)

结构化输出（简洁）

In [ ]:
# 一步到位
structured_llm = model.with_structured_output(Person)
person = structured_llm.invoke(" 张三是一名 30 岁的软件工程师 ", extra_body={"thinking": {"type": "disabled"}})
rprint(person)
# ✅ 自动解析、验证、创建对象

# 2、 四种模式的使用
2.1 模式 1 ： Pydantic

In [ ]:
from pydantic import BaseModel, Field


#定义 Pydantic 模型使用清晰的字段描述；没有描述， LLM 可能格式错误
class Person(BaseModel):
      """人物信息"""

      name: str = Field(description="姓名")
      age: int = Field(description="年龄")
      occupation: str = Field(description="职业")


#使用 with_structured_output 即可引导模型进行结构化输出： LangChain 会要求 LLM 的输出必须能填充这些字段。
# 创建结构化输出的 LLM
structured_llm = model.with_structured_output(Person)

# 调用
result = structured_llm.invoke(" 张三是一名 30 岁的软件工程师 ")

print(result)
print(type(result))

# result 是 Person 实例
print(result.name)  # " 张三 "
print(result.age)  # 30
print(result.occupation)  # " 软件工程师 "

## **举例 2：**

In [ ]:
from pydantic import BaseModel, Field, SecretStr


class MovieModel(BaseModel):
      """
      电影的详细信息
      """

      title: str = Field(description=" 电影标题 ")
      year: int = Field(description=" 电影上映年份 ")
      director: str = Field(description=" 导演 ")
      rating: float = Field(description=" 电影评分，满分十分 ")


model_with_structure = model.with_structured_output(MovieModel)
response = model_with_structure.invoke(" 给出盗梦空间的信息 ")
print(response)
print(type(response))

In [ ]:
## **举例 3：**

In [ ]:
from pydantic import BaseModel, Field


# 定义输出结构
class SentimentAnalysis(BaseModel):
      """情感分析结果"""

      sentiment: str = Field(description=" 情感倾向： positive/negative/neutral")
      confidence: float = Field(description=" 置信度， 0-1 之间 ")
      keywords: list[str] = Field(description=" 关键词列表 ")
      # ✅ v1.x ：使用 with_structured_output


structured_model = model.with_structured_output()
SentimentAnalysis

# 调用
text = " 这个课程内容很实用，学到了很多知识，强烈推荐！ "
result = structured_model.invoke(f" 分析以下文本的情感： \n{text}")

print(f" 类型 : {type(result)}")  # <class 'SentimentAnalysis'>
print(f" 情感 : {result.sentiment}")
print(f" 置信度 : {result.confidence}")
print(f" 关键词 : {result.keywords}")

### **情况 1：** 可选字段
**问题：** LLM 未填充某些字段怎么办？
使用 Optional 指定字段为可选的。
**举例：**

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional


class Person(BaseModel):
      """人物信息"""

      name: str = Field(description=" 姓名 ")
      age: int = Field(description=" 年龄 ")
      occupation: str = Field(description=" 职业 ")


structured_llm = model.with_structured_output(Person)

structured_llm.invoke(" 张三是一名医生 ")
Person(name=' 张三 ', age=0, occupation=' 医生 ')

对比

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field


class Person(BaseModel):
      """人物信息"""

      name: str = Field(description=" 姓名 ")
      age: Optional[int] = Field(description=" 年龄 ")
      occupation: str = Field(description=" 职业 ")


structured_llm = model.with_structured_output(Person)

structured_llm.invoke(" 张三是一名医生 ")

### **情况 2：** 默认值
LLM 未提供的信息会使用默认值。格式如下：
Field(default=" 默认值 ", description=" 描述 ")
**注意：** 不同模型提供商对 default 字段的支持是不同的。
**举例 1：**

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field


class Person(BaseModel):
      """人物信息"""

      name: str = Field(description=" 姓名 ")
      age: int = Field(1, description=" 年龄 ")
      occupation: str = Field(description=" 职业 ")


structured_llm = model.with_structured_output(Person)

structured_llm.invoke(" 张三是一名医生 ")

作为对比：

In [ ]:
class Config(BaseModel):
      timeout: Optional[int] = Field(30, description=" 超时时间 ( 单位秒 )")
      retry: bool = Field(False, description=" 是否支持重试 ")
      max_attempts: int = Field(6, description=" 最大重试次数 ")


# 测试
structured_llm = model.with_structured_output(Config)
structured_llm.invoke(" 配置要求：支持重试 , 最多重试 5 次 ")

**举例 3：**

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field


class Product(BaseModel):
      """产品信息"""

      name: str = Field(description=" 产品名称 ")
      price: float = Field(description=" 价格 ")
      description: Optional[str] = Field(description=" 产品描述 ")
      stock: int = Field(default=100, description=" 库存 ")


# 测试
structured_llm = model.with_structured_output(Product)
print("\n 场景 1 ：完整信息 ")
result1 = structured_llm.invoke("iPhone 15 售价 5999 元，最新款智能手机，库存 50台 ")
print(result1)

print("\n 场景 2 ：缺少描述和库存 ")
result2 = structured_llm.invoke("MacBook Pro 售价 12999 元 ")
print(result2)

### **情况 3：** 枚举类型
**问题：** 如何限制字段的可选值？
回答：使用枚举。
**举例 1：**

In [ ]:
from enum import Enum


class Priority(str, Enum):
      LOW = " 低 "
      MEDIUM = " 中 "
      HIGH = " 高 "


class Task(BaseModel):
      title: str
      priority: Priority  # 只能是 LOW/MEDIUM/HIGH

In [ ]:
from enum import Enum


class Status(str, Enum):
      ACTIVE = " 激活 "
      INACTIVE = " 未激活 "


class User(BaseModel):
      status: Status  # 只能是 ACTIVE 或 INACTIVE

In [ ]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field


# 定义你的优先级枚举类
class Priority(str, Enum):
      LOW = "低"
      MEDIUM = "中"
      HIGH = "高"


class CustomerInfo(BaseModel):
      """客户信息"""

      name: str = Field(description=" 客户姓名 ")
      phone: str = Field(description=" 电话号码 ")
      email: Optional[str] = Field(description=" 邮箱 ")
      issue: str = Field(description=" 问题描述 ")
      urgency: Priority = Field(description=" 紧急程度 ")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """
客服 : 您好，请问有什么可以帮助您？
客户 : 我是王小明，电话 138-1234-5678 ，我的订单一直没发货，很着急！
客服 : 好的，我帮您查一下
"""

result = structured_llm.invoke(f" 从以下客服对话中提取客户信息： \n{conversation}")

print(result)
print("\n 提取结果： ")
print(f" 客户 : {result.name}")
print(f" 电话 : {result.phone}")
print(f" 邮箱 : {result.email or ' 未提供 '}")
print(f" 问题 : {result.issue}")
print(f" 紧急程度 : {result.urgency.value}")

### 如果嫌单独定义一个 Enum 类太麻烦，也可以直接导入 typing 中的 Literal ，直接在字段里把允许
的值写死。

In [ ]:
from typing import Optional, Literal
from pydantic import BaseModel, Field


class CustomerInfo(BaseModel):
      """客户信息"""

      name: str = Field(description=" 客户姓名 ")
      phone: str = Field(description=" 电话号码 ")
      email: Optional[str] = Field(" 未提供 ", description=" 邮箱 ")
      issue: str = Field(description=" 问题描述 ")
      # 使用 Literal 直接限定字面量值
      urgency: Literal["低", "中", "高"] = Field(description=" 紧急程度 ")


# 测试
structured_llm = model.with_structured_output(CustomerInfo)

conversation = """
客服 : 您好，请问有什么可以帮助您？
客户 : 我是王小明，电话 138-1234-5678 ，我的订单一直没发货，很着急！
客服 : 好的，我帮您查一下
"""

result = structured_llm.invoke(f" 从以下客服对话中提取客户信息： \n{conversation}")

print(result)

print("\n 提取结果： ")
print(f" 客户 : {result.name}")
print(f" 电话 : {result.phone}")
print(f" 邮箱 : {result.email}")
print(f" 问题 : {result.issue}")
print(f" 紧急程度 : {result.urgency}")

### 应用场景：
自动填充 CRM 系统
工单自动分类
客服辅助
**情况 4：** 列表提取
**举例 1：**多个 Person 对象

In [ ]:
from typing import List


class Person(BaseModel):
      """人物信息"""

      name: str
      age: int


class PersonList(BaseModel):
      """人物列表信息"""

      people: List[Person]  # 多个 Person 对象


structured_llm = model.with_structured_output(PersonList)
result = structured_llm.invoke(" 张三 30 岁，李四 25 岁 ")

print(result)

**举例 2：** 产品评论分析

In [ ]:
class Review(BaseModel):
      """产品评论"""

      product: str
      rating: int = Field(description=" 评分 1-5")
      # pros: List[str] = Field(description=" 优点列表 ")
      pros: List[str] = Field(description="优点列表")
      cons: List[str] = Field(description=" 缺点列表 ")


structured_llm = model.with_structured_output(Review)

review = structured_llm.invoke(
      """
  iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。 4 分。
  """
)

print(review)

### 应用场景：
批量处理用户评论
自动生成分析报告
发现产品改进点
**举例 3：** 文档信息提取

In [ ]:
class Invoice(BaseModel):
      """发票信息"""

      invoice_number: str = Field(description=" 发票号 ")
      date: str = Field(description=" 日期 ")
      total_amount: float = Field(description=" 总金额 ")
      items: List[str] = Field(description=" 商品 ")


# 测试
structured_llm = model.with_structured_output(Invoice)

invoice_text = """
发票号 : INV-2024-001
日期 : 2024-01-15
总金额 : 1299.00
商品 : MacBook Pro, AppleCare+ """

invoice = structured_llm.invoke(f" 提取发票信息： {invoice_text}")
print(invoice)

### 应用场景：
自动化财务处理
OCR 后结构化
数据录入
**情况 5：** 嵌套结构
**举例 1：**

In [ ]:
from pydantic import BaseModel


class Address(BaseModel):
      """地点描述"""

      city: str
      district: str


class Company(BaseModel):
      """公司信息"""

      name: str
      address: Address  # 嵌套模型


structured_llm = model.with_structured_output(Company)
result = structured_llm.invoke(" 阿里巴巴在杭州滨江区 ")

print(result)

### LLM 能力有限，复杂嵌套结构可能会出错。所以建议：
嵌套层级 ≤ 3 层

In [ ]:
from pydantic import BaseModel, Field
from typing import List


# 1. 定义嵌套的 Pydantic 模型
class Actor(BaseModel):
      """演员信息"""

      name: str = Field(description=" 演员姓名 ")
      role: str = Field(description=" 饰演的角色 ")


class Movie(BaseModel):
      """电影信息"""

      title: str = Field(description=" 电影标题 ")
      year: int = Field(description=" 上映年份 ")
      director: str = Field(description=" 导演 ")
      cast: List[Actor] = Field(description=" 演员列表 ")  # 定义列表字段
      rating: float = Field(description=" 评分 ")


# 2. 初始化模型并绑定输出结构
structured_model = model.with_structured_output(Movie)

# 3. 调用模型，直接获取 Movie 实例
response = structured_model.invoke(" 请介绍电影《盗梦空间》 ")

# 4. 访问嵌套数据
print(f" 电影名 : {response.title}")
print(f" 上映年份 : {response.year}")
print(f" 导演 : {response.director}")
print(f" 演员列表 : {response.cast}")
print(f" 评分 : {response.rating}")

### 使用清晰的 description
必要时拆分成多个调用
**举例 3：**

In [ ]:
from pydantic import BaseModel
from typing import List


class Aspect(BaseModel):
      """评论维度"""

      name: str = Field(description=" 维度名称，如：质量、价格、服务 ")
      score: int = Field(description=" 评分， 1-5")
      comment: str = Field(description=" 具体评价 ")


class ProductReview(BaseModel):
      """产品评论分析"""

      overall_sentiment: str = Field(description=" 整体情感：positive/negative/neutral")
      overall_score: int = Field(description=" 综合评分， 1-5")
      aspects: List[Aspect] = Field(description=" 各维度评价 ")
      summary: str = Field(description=" 一句话总结 ")


# 创建结构化模型
structured_model = model.with_structured_output(ProductReview)
# 测试
review_text = """
这款笔记本电脑性能非常强大，运行大型软件毫无压力。
屏幕色彩鲜艳，看视频很舒服。
不过价格有点贵，而且风扇噪音较大。
客服态度很好，物流也快。
总体来说还是值得购买的。
"""

result = structured_model.invoke(f" 分析以下产品评论： \n{review_text}")

print(f" 整体情感 : {result.overall_sentiment}")
print(f" 综合评分 : {result.overall_score}/5")
print(f"\n 各维度评价 :")
for aspect in result.aspects:
      print(f" - {aspect.name}: {aspect.score}/5 - {aspect.comment}")
      print(f"\n 总结 : {result.summary}")

### **情况 6：** 限制条件
**举例 1：**

In [82]:
from pydantic import ValidationError


class User(BaseModel):
      name: str = Field(min_length=2, max_length=20)
      age: int = Field(ge=0, le=150)
      email: str

      print("\n 有效数据 :")
      try:
            user = User(name=" 张三 ", age=30, email="zhang@example.com")
            print(f"[OK] {user.name}, {user.age}, {user.email}")
      except ValidationError as e:
            print(f"[FAIL] {e}")

      print("\n 无效数据（年龄超出范围） :")
      try:
            user = User(name=" 李四 ", age=200, email="li@example.com")
            print(f"[OK] {user}")
      except ValidationError as e:
            print(f"[FAIL] 验证失败（符合预期） : {e.errors()[0]['msg']}")


 有效数据 :
[OK]  张三 , 30, zhang@example.com

 无效数据（年龄超出范围） :
[FAIL] 验证失败（符合预期） : Input should be less than or equal to 150


PydanticUserError: A non-annotated attribute was detected: `user = User(name=' 张三 ', age=30, email='zhang@example.com')`. All model fields require a type annotation; if `user` is not meant to be a field, you may be able to resolve this error by annotating it as a `ClassVar` or updating `model_config['ignored_types']`.

For further information visit https://errors.pydantic.dev/2.12/u/model-field-missing-annotation